# Network Momentum replication — Stage A validation

This notebook is the research/QA entry point. Method logic lives in `src/`. The current run validates the locked inputs and builds the causal eight-dimensional $U_t$; it does **not** run graph learning or a backtest.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from network_momentum.backtest import run_stage_a
from network_momentum.config import DEFAULT_CONFIG, UNDOCUMENTED_CHOICES
from network_momentum.data_utils import load_model_inputs

pd.set_option('display.max_columns', 20)
print('Project root:', PROJECT_ROOT)

## Configuration and explicit assumptions

Paper-specified values and unresolved implementation choices are centralized in `src/config.py`.

In [ ]:
display(pd.Series(UNDOCUMENTED_CHOICES, name='current_setting').to_frame())
DEFAULT_CONFIG.features

## Module 0 — locked input inspection

Returns and availability are authoritative. Prices are loaded only for MACD. The long model-data table is an audit source.

In [ ]:
inputs = load_model_inputs(DEFAULT_CONFIG.paths)
print('returns:', inputs.returns.shape, inputs.returns.index.min().date(), 'to', inputs.returns.index.max().date())
print('availability exact:', inputs.availability.equals(inputs.returns.notna()))
display(inputs.returns.notna().sum().sort_values().to_frame('return_observations').head(10))

In [ ]:
representative = ['CL', 'ES', 'TY', 'GC']
available = [asset for asset in representative if asset in inputs.prices.columns]
inputs.prices[available].plot(subplots=True, figsize=(11, 8), title='Supporting continuous prices (no fill)')
plt.tight_layout();

## Modules 1–3 — volatility, eight features, causal winsorization

This cell runs the modular Stage-A pipeline, writes auditable artifacts under `results/replication/artifacts/`, and returns all QA reports.

In [ ]:
artifacts, reports = run_stage_a(DEFAULT_CONFIG, save=True)
display(pd.json_normalize(reports['input_qa']).T.rename(columns={0: 'value'}).head(20))
display(pd.Series(reports['long_model_data_audit'], name='value').to_frame())
display(pd.Series(reports['feature_qa'], name='value').to_frame())

## Inspect $U_t$ and timing coverage

At one date, rows are assets and columns are exactly eight individual momentum features. No future-return target is stored in this matrix.

In [ ]:
complete = artifacts.u_winsorized.dropna()
sample_date = complete.index.get_level_values('date').max()
U_t = artifacts.u_winsorized.xs(sample_date, level='date').dropna()
print('feature date = signal information date =', sample_date.date())
print('U_t shape:', U_t.shape)
display(U_t.head())
assert U_t.shape[1] == 8

In [ ]:
complete_by_date = artifacts.u_winsorized.notna().all(axis=1).groupby(level='date').sum()
complete_by_date.plot(figsize=(11, 3), title='Assets with complete 8D U_t by date')
plt.ylabel('assets')
plt.tight_layout();

In [ ]:
asset = 'CL' if 'CL' in artifacts.u_winsorized.index.get_level_values('asset') else U_t.index[0]
artifacts.u_winsorized.xs(asset, level='asset').plot(subplots=True, figsize=(11, 12), title=f'Winsorized U_t features — {asset}')
plt.tight_layout();

## Stop checkpoint

The validated `U_winsorized.parquet` is the current milestone. Next is a **single-date, single-lookback graph prototype**. Do not launch the full graph history or walk-forward backtest until Stage-A coverage and representative feature paths are reviewed.